## Step 1: Set Up the Notebook

Install the SDK in the first cell.

In [ ]:
!pip install -q olostep

The second cell loads your API key. It checks the `OLOSTEP_API_KEY` environment variable first and falls back to a hidden prompt, so the key never lands in the notebook.

In [ ]:
import os
from getpass import getpass

from olostep import Olostep  # make sure you've installed the Python SDK

# 1. Retrieve the API key
api_key = os.getenv("OLOSTEP_API_KEY")
if not api_key:
    api_key = getpass("Please enter your Olostep API Key: ")
    os.environ["OLOSTEP_API_KEY"] = api_key

# 2. Initialize the client
client = Olostep(api_key=api_key)

The third cell holds the settings you'll change for your own project.

In [ ]:
SITE = "https://stripe.com"  # your site
BRAND = "Stripe"  # how people write your brand name
BRAND_DOMAIN = "stripe.com"  # your bare domain
BLOG_PATH = "/blog/"  # where your site publishes articles
CATEGORY = "payment processing platforms for online businesses"  # the topic you want to rank and be cited for

COMPETITOR = "https://wise.com"  # Wise as the new competitor
COMPETITOR_START = "https://wise.com/blog/"  # where Wise lists its articles
COMPETITOR_PATTERN = "/blog/**"  # URL pattern for those articles

This tutorial uses Stripe as the site and Adyen as the competitor. Both are examples, and the notebook only reports what Olostep returns. Every later cell reads these values, so replace them with your own site and a competitor of yours.

## Step 2: Map Your Site Structure

A map returns the URLs Olostep finds on a site. For an SEO team, that is the raw material of a content inventory.

In [ ]:
site_map = client.maps.create(url=SITE)
urls = list(site_map.urls())
print(f"Found {len(urls)} URLs")

`site_map.urls()` yields each URL as a string, and `list()` collects them. Counting URLs by their first path segment shows where your site's weight sits.

In [ ]:
from collections import Counter
from urllib.parse import urlparse


def section(url):
    first = urlparse(url).path.strip("/").split("/")[0]
    return f"/{first}" if first else "/"


for name, count in Counter(section(u) for u in urls).most_common(10):
    print(f"{count:>5}  {name}")

The loop prints your ten largest sections. A section with hundreds of URLs and no internal links pointing to it is worth a look, and so is a section you expected to be large that shows up with a handful. You'll reuse `urls` in Step 7.

## Step 3: Find Competitors with Search

Search takes a natural-language query and returns deduplicated links with titles and descriptions. Pass your own domain to `exclude_domains` so you don't find yourself.

In [ ]:
search = client.searches.create(
    f"best {CATEGORY}",
    limit=10,
    exclude_domains=[BRAND_DOMAIN],
)

for link in search.links:
    print(urlparse(link["url"]).netloc, "|", link["title"])

Each link is a dictionary with `url`, `title` and `description`. A category query returns listicles and review sites next to direct competitors, so read the list and pick a real one. Each search costs 5 credits, per the [search docs](https://docs.olostep.com/features/search). The rest of the tutorial studies Wise, which the config cell sets in `COMPETITOR`. If you use your own site, paste a competitor from this list into that value.

In [ ]:
page = client.scrapes.create(url_to_scrape=COMPETITOR, formats=["markdown"])
markdown = page.markdown_content

print(f"{len(markdown.split())} words")
for line in markdown.splitlines():
    if line.startswith("#"):
        print(line)

The heading lines are the page's outline. They show which topics the competitor puts first, and how many words they spend on them.

## Step 5: Crawl a Competitor's Articles

A scrape handles one page. A crawl starts at a URL and follows links, and `include_urls` limits it to the paths you care about. `max_pages=25` keeps this run small.

Wise publishes its articles under `/blog/`, so the config cell points the crawl there. On your own competitor, find the section that lists their articles and set `COMPETITOR_START` and `COMPETITOR_PATTERN` to match.

In [ ]:
crawl = client.crawls.create(
    start_url=COMPETITOR_START,
    include_urls=[COMPETITOR_PATTERN],
    max_pages=25,
)

This tutorial uses Stripe as the site and Wise as the competitor. Both are examples, and the notebook only reports what Olostep returns. Every later cell reads these values, so replace them with your own site and a competitor of yours.

## Step 2: Map Your Site Structure

A map returns the URLs Olostep finds on a site. For an SEO team, that is the raw material of a content inventory.

In [ ]:
rows = []
for post in crawl.pages():
    content = post.retrieve(["markdown"])
    words = len((content.markdown_content or "").split())
    rows.append((post.url, words))

for url, words in sorted(rows, key=lambda r: r[1], reverse=True)[:10]:
    print(f"{words:>6}  {url}")

You now have a ranked list of the competitor's longest posts. The URL slugs show the topics they cover, and the word counts show where they go deep.

## Step 6: Pull a Competitor Profile with Answers

Some facts are not on one page: what a company sells, who it targets, how it prices. Answers browses the web for them and returns a result with sources. Pass `json_format` to get named fields back.

In [ ]:
answer = client.answers.create(
    task=f"What does {COMPETITOR} sell, who is it for, and how is it priced?",
    json_format={"product": "", "audience": "", "pricing": ""},
)

**But what happens when you index into the result?**

**The problem:** the cell stops with `TypeError: string indices must be integers, not 'str'`. In the SDK, `answer.json_content` is a JSON string, so text keys don't work on it.

**The solution:** parse it with `json.loads`.

In [ ]:
import json

profile = json.loads(answer.json_content)
print(profile)
print(answer.sources)

`profile` is now a dictionary with your three fields, and `answer.sources` lists the URLs the answer came from. Fields Olostep can't verify come back as `NOT_FOUND`.

An answer costs 20 credits, per the [answers docs](https://docs.olostep.com/features/answers). I'd use it for the few facts you can't scrape, and use Step 4 for anything that sits on a page.

## Step 7: Scrape a URL List in Bulk with Batches

Crawls and batches both collect many pages, and they suit different starting points. Use a crawl when you have a start URL and a path pattern. Use a batch when you already have the list of URLs, like the `urls` from Step 2.

This cell audits your own blog for thin pages.

In [ ]:
blog_urls = [u for u in urls if BLOG_PATH in u][:100]

batch = client.batches.create(blog_urls)

The GEO docs say new accounts are limited to 100 items per batch, which is why the list stops at 100. Higher limits are available on request.

In [ ]:
thin = []
for item in batch.items():  # waits for the batch to finish
    content = item.retrieve(["markdown"])
    words = len((content.markdown_content or "").split())
    if words < 300:
        thin.append((item.url, words))

print(f"{len(thin)} of {len(blog_urls)} posts are under 300 words")
for url, words in sorted(thin, key=lambda r: r[1])[:10]:
    print(f"{words:>6}  {url}")

The 300-word cutoff is my choice, so adjust it to your content. The posts at the top of this list are your candidates to expand, merge or redirect.

## Step 8: Track Your Brand Across AI Engines

This is the step that makes the notebook a visibility tool. Olostep's AI search parsers send a prompt to an engine and return structured JSON: the answer as markdown, and the sources it used.

Each batch uses one parser, and you put the prompt in the engine's URL template. Define the engines and your prompts first.

In [ ]:
from urllib.parse import quote_plus

ENGINES = {
    "chatgpt": ("@olostep/chatgpt-results", "https://chatgpt.com/?q={q}"),
    "perplexity": ("@olostep/perplexity-results", "https://www.perplexity.ai/?q={q}"),
    "gemini": ("@olostep/gemini-results", "https://gemini.google.com/?q={q}"),
    "ai_mode": ("@olostep/google-aimode-results", "https://www.google.com/aimode?q={q}"),
    "ai_overview": ("@olostep/google-ai-overview-results", "https://www.google.com/search?q={q}"),
    "copilot": ("@olostep/microsoft-copilot-results", "https://copilot.microsoft.com/chats?q={q}"),
}

PROMPTS = [
    f"best {CATEGORY}",
    f"which {CATEGORY} do developers recommend",
]

Replace the two demo prompts with the questions your customers ask. Now start one batch per engine. Each item gets a `custom_id`, so you can match results back to prompts.

In [ ]:
batches = {}
for engine, (parser, template) in ENGINES.items():
    batches[engine] = client.batches.create(
        urls=[
            {"custom_id": str(i), "url": template.format(q=quote_plus(prompt))}
            for i, prompt in enumerate(PROMPTS)
        ],
        parser=parser,
        country="US",
    )

Starting every batch before reading any of them lets the engines run side by side. The GEO docs put processing at roughly 20 to 25 minutes, so this is a good moment to make tea. ChatGPT items cost 5 credits each and the other engines cost 3, per the [GEO docs](https://docs.olostep.com/examples/geo). Six engines and two prompts come to 40 credits.

Read each result and score it twice. A **mention** means your brand name appears in the answer text. A **citation** means one of the sources is on your domain.

In [ ]:
rows = []
cited_domains = Counter()

for engine, batch in batches.items():
    for item in batch.items():  # waits for the batch to finish
        data = item.retrieve(["json"]).json_content or {}
        answer_text = (data.get("answer_markdown") or "").lower()
        sources = [s.get("url", "") for s in data.get("sources") or []]

        cited_domains.update(urlparse(s).netloc for s in sources)
        rows.append({
            "engine": engine,
            "prompt": PROMPTS[int(item.custom_id)],
            "mentioned": BRAND.lower() in answer_text,
            "cited": any(BRAND_DOMAIN in s for s in sources),
            "sources": len(sources),
        })

Load the rows into a DataFrame and average the two flags per engine.

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
print(df.groupby("engine")[["mentioned", "cited"]].mean().round(2))

A value of `1.0` means every prompt produced a mention or a citation on that engine, and `0.0` means none did. The two columns tell different stories. A mention without a citation means the engine knows your name and did not use your page. A citation without a mention means it used your page and did not name you.

The last cell counts which domains the engines cite for your category.

In [ ]:
for domain, count in cited_domains.most_common(10):
    print(f"{count:>4}  {domain}")

These are the domains AI engines pull from when they answer your category. They can differ from the competitors you found in Step 3, and they are the pages you'd study first.

## Run It Again

One run is a snapshot. Save `df` to a CSV with the run date in the filename, rerun the notebook on a schedule you choose, and compare the averages over time. Olostep also has a monitors endpoint that watches pages or queries on a schedule and sends change alerts, if you want to move the tracking out of a notebook.

## Next Steps

You now have a notebook that maps your site, finds competitors, audits their content, builds a profile and measures how often six AI engines mention and cite your brand. From here, swap the two demo prompts for your real ones, add competitor names to the mention check, and run the notebook weekly.

To go further, read the [Olostep GEO example](https://docs.olostep.com/examples/geo) for the full output fields per engine, including ads and shopping cards.